In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output
import country_converter as coco
import re
import numpy as np
import time

# ---  Load  data ---
file_path = "Scale_up_output_MS.csv"
df = pd.read_csv(file_path)

# Rename first column 
df.rename(columns={df.columns[0]: "Country"}, inplace=True)

# --- Safe cleaning function (keep exponent with e+03 etc.) ---
def clean_value(x):
    if isinstance(x, str):
        # Case: (2.0+/-1.1)e+05
        match = re.match(r"\(([\d\.]+)\+/-.*?\)(e[+-]?\d+)?", x)
        if match:
            base = match.group(1)
            exp = match.group(2) if match.group(2) else ""
            try:
                return float(base + exp)
            except:
                return np.nan
        # Case: 0.0+/-0
        match2 = re.match(r"([\d\.]+)\+/-", x)
        if match2:
            try:
                return float(match2.group(1))
            except:
                return np.nan
        # Case: plain number
        try:
            return float(x)
        except:
            return np.nan
    return x

week_cols = df.columns[2:]  
df[week_cols] = df[week_cols].applymap(clean_value)

df = df.copy()

# --- Accumulate airflow across weeks ---
df[week_cols] = df[week_cols].cumsum(axis=1)

# --- Add UN region + subregion + ISO3 ---
cc = coco.CountryConverter()
df["Region"] = cc.convert(names=df["Country"], to="UNregion")
df["Subregion"] = cc.convert(names=df["Country"], to="continent")  # fallback
df["ISO3"] = cc.convert(names=df["Country"], to="ISO3")

df_melt = df.melt(
    id_vars=["Country", "ISO3", "Region", "Subregion"],
    value_vars=week_cols,
    var_name="Week", value_name="AirFlow"
)

# --- color scale ---
vmin = df[week_cols].min().min()
vmax = df[week_cols].max().max()

# ---  Widgets ---
week_index = widgets.IntSlider(
    value=1,
    min=1,
    max=len(week_cols),
    step=1,
    description="Week:"
)

group_choice = widgets.Dropdown(
    options=["Region", "Subregion"],
    value="Region",
    description="Group by:"
)

region_filter = widgets.Dropdown(
    options=["All"] 
            + sorted(df["Region"].dropna().unique().tolist()) 
            + sorted(df["Subregion"].dropna().unique().tolist()),
    value="All",
    description="Filter:"
)

play_button = widgets.Button(
    description="▶ Play",
    tooltip="Auto-play weeks",
    button_style="success"
)

out = widgets.Output()

# --- Plotting ---
def update_plots(change=None):
    week = str(week_index.value)
    group = group_choice.value
    filter_value = region_filter.value
    
    with out:
        clear_output(wait=True)

        # Filter dataframe if region filter applied
        if filter_value != "All":
            df_filtered = df[(df["Region"] == filter_value) | (df["Subregion"] == filter_value)]
        else:
            df_filtered = df

        # Choropleth for cumulative airflow
        fig1 = px.choropleth(
            df_filtered,
            locations="ISO3",          
            locationmode="ISO-3",
            color=week,
            hover_name="Country",
            color_continuous_scale=[(0, "white"), (1, "darkgreen")],
            range_color=[vmin, vmax],
            title=f"Total Air Flow by Country (up to Week {week})"
        )
        fig1.show()

        # Historical line plot 
        fig2 = px.line(
            df_melt,
            x="Week", y="AirFlow", color=group,
            title=f"Total Air Flow by UN {group}"
        )
        fig2.update_layout(
            plot_bgcolor="#E6FFE6",   # chart area
        )
        fig2.show()

# ---  button logic ---
def play_clicked(b):
    for w in range(1, len(week_cols) + 1):
        week_index.value = w
        time.sleep(0.5)  # adjust playback speed 

play_button.on_click(play_clicked)


week_index.observe(update_plots, names="value")
group_choice.observe(update_plots, names="value")
region_filter.observe(update_plots, names="value")

from IPython.display import HTML
from ipywidgets import VBox

custom_style = """
<style>
#custom-container {
    background-color: #fffacd;   /* light yellow */
    padding: 15px;
    border-radius: 10px;
    margin-top: 10px;
    margin-bottom: 20px;
}
#custom-banner {
    width: 100%;
    background-color: #006400;   /* dark green */
    color: white;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    padding: 10px;
    border-radius: 6px;
    margin-bottom: 15px;
}
</style>
"""
display(HTML(custom_style))


container = widgets.VBox([
    widgets.HTML('<div id="custom-banner">ALLFED</div>'),
    week_index,
    group_choice,
    region_filter,
    play_button,
    out
])
container.layout = widgets.Layout(width="100%", padding="15px", border="solid 2px #006400", border_radius="10px", background_color="#fffacd")

display(container)


update_plots()


C:\Users\26959\AppData\Local\Temp\ipykernel_136176\1907702008.py:44: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

C:\Users\26959\AppData\Local\Temp\ipykernel_136176\1907702008.py:53: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\26959\AppData\Local\Temp\ipykernel_136176\1907702008.py:54: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\26959\AppData\Local\Temp\ipykernel_136176\1907702008.py:55: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result o